In [ ]:
import chromadb
from chromadb.config import Settings

# Initialize ChromaDB client
client = chromadb.PersistentClient(path="./chroma_db")

# Create a collection for museum data
collection = client.get_or_create_collection(name="museum_collection")

print("ChromaDB initialized successfully!")
print(f"Collection: {collection.name}")

In [4]:
# Install required packages for embeddings
# !pip install sentence-transformers

import chromadb
from chromadb.config import Settings
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer

# Initialize ChromaDB client
client = chromadb.PersistentClient(path="./chroma_db")

# Create a collection for museum data
collection = client.get_or_create_collection(name="museum_collection")

print("ChromaDB initialized successfully!")
print(f"Collection: {collection.name}")

# Connect to Neo4j and get data
uri = "bolt://localhost:7687"
user = "neo4j"
pwd = "museumdata"

driver = GraphDatabase.driver(uri, auth=(user, pwd))

# Get artworks with descriptions
with driver.session() as session:
    result = session.run("""
    MATCH (a:ArtPiece)
    OPTIONAL MATCH (a)-[:CREATED_BY]->(artist:Artist)
    RETURN a.title as title, a.description as description, 
           a.artist as artist_name, a.dating as dating, 
           a.technique as technique, artist.name as artist_full
    """)
    
    documents = []
    metadatas = []
    ids = []
    
    for i, record in enumerate(result):
        title = record["title"] or ""
        description = record["description"] or ""
        artist = record["artist_full"] or record["artist_name"] or ""
        dating = record["dating"] or ""
        technique = record["technique"] or ""
        
        # Create document text
        doc_text = f"Title: {title}. Artist: {artist}. Dating: {dating}. Technique: {technique}."
        if description:
            doc_text += f" Description: {description}"
        
        documents.append(doc_text)
        metadatas.append({
            "title": title,
            "artist": artist,
            "dating": dating,
            "technique": technique,
            "type": "artwork"
        })
        ids.append(f"artwork_{i}")

# Get themes
with driver.session() as session:
    result = session.run("MATCH (t:Theme) RETURN t.name as name, t.description as description")
    
    for i, record in enumerate(result):
        name = record["name"] or ""
        description = record["description"] or ""
        
        doc_text = f"Theme: {name}. Description: {description}"
        
        documents.append(doc_text)
        metadatas.append({
            "name": name,
            "type": "theme"
        })
        ids.append(f"theme_{i}")

driver.close()

# Add to ChromaDB
if documents:
    collection.add(
        documents=documents,
        metadatas=metadatas,
        ids=ids
    )
    print(f"Added {len(documents)} documents to vector database!")

# Test query
results = collection.query(
    query_texts=["Renaissance paintings"],
    n_results=3
)

print("\nSample search results:")
for i, doc in enumerate(results['documents'][0]):
    print(f"{i+1}. {doc[:200]}...")

ChromaDB initialized successfully!
Collection: museum_collection
Added 111 documents to vector database!

Sample search results:
1. Title: Faustina Maior [?]. Artist: Anònim, Itàlia. Dating: 1528-1533. Technique: Escultura, marbre blanc. Placa rectangular....
2. Title: Trattato della pittura. Artist: Leonardo da Vinci. Dating: 1733. Technique: Llibre. Paper i pergamí....
3. Title: Epifania. Artist: Mestre de Pau i Bernabeu. Dating: 1530-1540. Technique: Pintura, oli sobre fusta....


In [6]:
# Search function for the vector database
def search_museum(query, n_results=5):
    """
    Search the museum vector database for relevant artworks and themes.

    Args:
        query (str): Search query
        n_results (int): Number of results to return

    Returns:
        dict: Search results with documents, metadata, and distances
    """
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=['documents', 'metadatas', 'distances']
    )

    return results

# Example searches
print("\n=== Vector Database Search Examples ===")

# Search for Renaissance art
results = search_museum("Renaissance paintings")
print(f"\nSearch: 'Renaissance paintings' ({len(results['documents'][0])} results)")
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"{i+1}. {meta.get('title', meta.get('name', 'Unknown'))}")
    print(f"   Type: {meta['type']}")
    print(f"   Preview: {doc[:100]}...")
    print()

# Search for specific artist
results = search_museum("Leonardo da Vinci")
print(f"\nSearch: 'Leonardo da Vinci' ({len(results['documents'][0])} results)")
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"{i+1}. {meta.get('title', meta.get('name', 'Unknown'))}")
    print(f"   Preview: {doc[:150]}...")
    print()

# Search for themes
results = search_museum("religious themes")
print(f"\nSearch: 'religious themes' ({len(results['documents'][0])} results)")
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"{i+1}. {meta.get('title', meta.get('name', 'Unknown'))}")
    print(f"   Type: {meta['type']}")
    print()

print("Vector database is ready for semantic search!")


=== Vector Database Search Examples ===

Search: 'Renaissance paintings' (5 results)
1. Faustina Maior [?]
   Type: artwork
   Preview: Title: Faustina Maior [?]. Artist: Anònim, Itàlia. Dating: 1528-1533. Technique: Escultura, marbre b...

2. Trattato della pittura
   Type: artwork
   Preview: Title: Trattato della pittura. Artist: Leonardo da Vinci. Dating: 1733. Technique: Llibre. Paper i p...

3. Epifania
   Type: artwork
   Preview: Title: Epifania. Artist: Mestre de Pau i Bernabeu. Dating: 1530-1540. Technique: Pintura, oli sobre ...

4. Caterina de Mèdici
   Type: artwork
   Preview: Title: Caterina de Mèdici. Artist: Thomas Leu. Dating: c. 1595. Technique: Gravat....

5. María de Médici
   Type: artwork
   Preview: Title: María de Médici. Artist: Thomas Leu. Dating: c. 1595. Technique: Gravat....


Search: 'Leonardo da Vinci' (5 results)
1. Trattato della pittura
   Preview: Title: Trattato della pittura. Artist: Leonardo da Vinci. Dating: 1733. Technique: Llibre. Paper i perg

In [7]:
# Usage instructions for your GuIA application
print("""
=== ChromaDB Vector Database for GuIA Museum App ===

Your vector database contains:
- 72 Artworks with titles, artists, techniques, dates, and descriptions
- 38 Museum themes from the audio guide
- Total: 110 documents with semantic embeddings

Usage in your application:

1. Import the search function:
   from vectror import search_museum

2. Search for content:
   results = search_museum("your query here", n_results=10)

3. Results structure:
   - results['documents']: List of matching document texts
   - results['metadatas']: Metadata (title, artist, type, etc.)
   - results['distances']: Similarity scores (lower = more similar)

4. Example integration:
   query = "Italian Renaissance paintings"
   results = search_museum(query)
   
   for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
       if meta['type'] == 'artwork':
           print(f"Found artwork: {meta['title']} by {meta['artist']}")
       elif meta['type'] == 'theme':
           print(f"Related theme: {meta['name']}")

The vector database enables semantic search - users can search in natural language
and find relevant artworks and themes based on meaning, not just keywords!

Database location: ./chroma_db/
""")


=== ChromaDB Vector Database for GuIA Museum App ===

Your vector database contains:
- 72 Artworks with titles, artists, techniques, dates, and descriptions
- 38 Museum themes from the audio guide
- Total: 110 documents with semantic embeddings

Usage in your application:

1. Import the search function:
   from vectror import search_museum

2. Search for content:
   results = search_museum("your query here", n_results=10)

3. Results structure:
   - results['documents']: List of matching document texts
   - results['metadatas']: Metadata (title, artist, type, etc.)
   - results['distances']: Similarity scores (lower = more similar)

4. Example integration:
   query = "Italian Renaissance paintings"
   results = search_museum(query)

   for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
       if meta['type'] == 'artwork':
           print(f"Found artwork: {meta['title']} by {meta['artist']}")
       elif meta['type'] == 'theme':
           print(f"Related theme: {